kubectl port-forward svc/mlflow-mlflow  -n mldata 5000:5000

In [22]:
import os
import sys
import mlflow
import kubernetes
from kubernetes.client.rest import ApiException
from dotenv import load_dotenv

load_dotenv()

True

In [23]:
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
NAMESPACE = os.getenv("KSERVE_NAMESPACE", "ml-serving")
SERVICE_NAME = os.getenv("KSERVE_SERVICE_NAME", "mobile-sales-predictor")
MODEL_NAME = os.getenv("MODEL_NAME", "mobile-sales-predictor")
ALIAS_CHAMPION = os.getenv("MODEL_ALIAS", "champion")

In [24]:
# Set MLflow tracking URI
mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.tracking.MlflowClient()


In [25]:
champion_version = client.get_model_version_by_alias(MODEL_NAME, ALIAS_CHAMPION)
champion_version.version

'16'

In [27]:
model_uri = f"runs:/{champion_version.run_id}/model"
model_uri

'runs:/43eb786588eb4c33857198418f4fca3a/model'

In [28]:
kubernetes.config.load_kube_config()  
api = kubernetes.client.CustomObjectsApi()
group = "serving.kserve.io"
version = "v1beta1"
plural = "inferenceservices"

In [29]:
isvc = api.get_namespaced_custom_object(group, version, NAMESPACE, plural, SERVICE_NAME)
isvc

{'apiVersion': 'serving.kserve.io/v1beta1',
 'kind': 'InferenceService',
 'metadata': {'annotations': {'kubectl.kubernetes.io/last-applied-configuration': '{"apiVersion":"serving.kserve.io/v1beta1","kind":"InferenceService","metadata":{"annotations":{"serving.kserve.io/s3-endpoint":"mlminio.mldata.svc.cluster.local:9000","serving.kserve.io/s3-usehttps":"false"},"name":"mobile-sales-predictor","namespace":"ml-serving"},"spec":{"predictor":{"containers":[{"command":["mlflow","models","serve","--model-uri","$(MLFLOW_S3_ARTIFACT_URI)","--host","0.0.0.0","--port","8080","--env-manager","local"],"env":[{"name":"MLFLOW_S3_ENDPOINT_URL","value":"http://mlminio.mldata.svc.cluster.local:9000"},{"name":"AWS_ACCESS_KEY_ID","valueFrom":{"secretKeyRef":{"key":"awsAccessKeyID","name":"minio-credentials"}}},{"name":"AWS_SECRET_ACCESS_KEY","valueFrom":{"secretKeyRef":{"key":"awsSecretAccessKey","name":"minio-credentials"}}},{"name":"MLFLOW_S3_ARTIFACT_URI","value":"s3://mlflow-artifacts/1/cd1a918f04534

In [30]:
containers = isvc["spec"]["predictor"]["containers"]
for idx, container in enumerate(containers):
    print(f"Container [{idx}]: {container.get('name')}")
  
    env_vars = container.get("env", [])
    
    for env in env_vars:
        name = env.get("name")
        value = env.get("value")
        print(f"  Env: {name} = {value}")

Container [0]: kserve-container
  Env: MLFLOW_S3_ENDPOINT_URL = http://mlminio.mldata.svc.cluster.local:9000
  Env: AWS_ACCESS_KEY_ID = None
  Env: AWS_SECRET_ACCESS_KEY = None
  Env: MLFLOW_S3_ARTIFACT_URI = runs:/757d23c7fadf454da35e7fc8f9640025/model


In [31]:
containers = isvc["spec"]["predictor"]["containers"]
for container in containers:
    for env in container.get("env", []):
        if env["name"] == "MLFLOW_S3_ARTIFACT_URI":
            print(env["value"])
            env["value"] = model_uri
            print(env["value"])


runs:/757d23c7fadf454da35e7fc8f9640025/model
runs:/43eb786588eb4c33857198418f4fca3a/model


In [32]:
api.replace_namespaced_custom_object(group, version, NAMESPACE, plural, SERVICE_NAME, isvc)

{'apiVersion': 'serving.kserve.io/v1beta1',
 'kind': 'InferenceService',
 'metadata': {'annotations': {'kubectl.kubernetes.io/last-applied-configuration': '{"apiVersion":"serving.kserve.io/v1beta1","kind":"InferenceService","metadata":{"annotations":{"serving.kserve.io/s3-endpoint":"mlminio.mldata.svc.cluster.local:9000","serving.kserve.io/s3-usehttps":"false"},"name":"mobile-sales-predictor","namespace":"ml-serving"},"spec":{"predictor":{"containers":[{"command":["mlflow","models","serve","--model-uri","$(MLFLOW_S3_ARTIFACT_URI)","--host","0.0.0.0","--port","8080","--env-manager","local"],"env":[{"name":"MLFLOW_S3_ENDPOINT_URL","value":"http://mlminio.mldata.svc.cluster.local:9000"},{"name":"AWS_ACCESS_KEY_ID","valueFrom":{"secretKeyRef":{"key":"awsAccessKeyID","name":"minio-credentials"}}},{"name":"AWS_SECRET_ACCESS_KEY","valueFrom":{"secretKeyRef":{"key":"awsSecretAccessKey","name":"minio-credentials"}}},{"name":"MLFLOW_S3_ARTIFACT_URI","value":"s3://mlflow-artifacts/1/cd1a918f04534